In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import networkx as nx
from shapely.geometry import Point, Polygon, MultiPolygon, LineString
import random
from datetime import datetime, timedelta
import os

In [2]:
# --- Simulation Parameters ---
NUM_SIMULATED_USERS = 50  # Increased for more data diversity
NUM_SIMULATION_DAYS = 60  # Increased for longer sequences
PLACE_NAME = "Da Nang, Vietnam"
OUTPUT_FILENAME = "simulated_movement_data.parquet"

# --- Define POI Categories for Different Personas ---
tags_for_fetch = {
    "amenity": ["school", "restaurant", "food_court", "cafe", "university"],
    "building": ["school", "apartments", "residential", "house", "dormitory", "office", "hotel"],
    "shop": ["mall", "department_store"],
    "leisure": ["park", "playground", "garden", "stadium"],
    "tourism": ["hotel", "museum", "attraction"]
}
tags_all_buildings = {"building": True}

# --- Create Output Directory ---
if not os.path.exists("./HistoricalMovement"):
    os.makedirs("./HistoricalMovement")
print("Configuration complete.")

Configuration complete.


In [3]:
ox.settings.use_cache = True
ox.settings.log_console = True

try:
    print(f"Geocoding '{PLACE_NAME}'...")
    place_polygon = ox.geocode_to_gdf(PLACE_NAME).unary_union
    
    print("Fetching road network...")
    road_network = ox.graph_from_polygon(place_polygon, network_type="drive", simplify=True)
    road_network_proj = ox.project_graph(road_network)
    nodes_proj, _ = ox.graph_to_gdfs(road_network_proj, nodes=True, edges=True)
    
    print("\nFetching all buildings...")
    all_buildings_gdf = ox.features_from_polygon(place_polygon, tags_all_buildings)
    
    print("Fetching all other POIs...")
    all_pois_gdf = ox.features_from_polygon(place_polygon, tags_for_fetch)
    
    print("All GIS data loaded successfully.")
except Exception as e:
    print(f"Could not fetch OSM data. Error: {e}")
    exit()

Geocoding 'Da Nang, Vietnam'...
Fetching road network...


/tmp/ipykernel_46942/1731171409.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  place_polygon = ox.geocode_to_gdf(PLACE_NAME).unary_union



Fetching all buildings...
Fetching all other POIs...
All GIS data loaded successfully.


In [4]:
def filter_and_centroid(gdf, tags_dict):
    """Filters a GeoDataFrame and converts geometries to centroids."""
    masks = []
    for key, values in tags_dict.items():
        if key in gdf.columns:
            base_mask = gdf[key].notna()
            if isinstance(values, list):
                masks.append(base_mask & gdf[key].isin(values))
            else:
                masks.append(base_mask & (gdf[key] == values))
    
    if not masks: return gpd.GeoDataFrame()
    
    combined_mask = pd.concat(masks, axis=1).any(axis=1)
    filtered_gdf = gdf[combined_mask]
    
    if filtered_gdf.empty: return gpd.GeoDataFrame()

    gdf_copy = filtered_gdf[filtered_gdf.geometry.is_valid & ~filtered_gdf.geometry.is_empty].copy()
    gdf_copy['geometry'] = gdf_copy['geometry'].centroid
    return gdf_copy

# --- Create GeoDataFrames for each persona's potential locations ---
pois_school = filter_and_centroid(all_pois_gdf, {"amenity": ["school", "university"], "building": "school"})
pois_home_candidates = filter_and_centroid(all_buildings_gdf, {"building": ["apartments", "residential", "house"]})
pois_office = filter_and_centroid(all_buildings_gdf, {"building": "office"})
pois_leisure = filter_and_centroid(all_pois_gdf, {"leisure": ["park", "playground", "garden", "stadium"]})
pois_commercial = filter_and_centroid(all_pois_gdf, {"shop": ["mall", "department_store"]})
pois_food = filter_and_centroid(all_pois_gdf, {"amenity": ["restaurant", "food_court", "cafe"]})
pois_hotel = filter_and_centroid(all_pois_gdf, {"tourism": "hotel", "building": "hotel"})

# --- Fallbacks to ensure data generation ---
pois_home = pois_home_candidates if not pois_home_candidates.empty else to_centroids(all_buildings_gdf)
pois_office = pois_office if not pois_office.empty else to_centroids(all_buildings_gdf.sample(frac=0.2, random_state=1))

print(f"Found: {len(pois_school)} schools, {len(pois_home)} homes, {len(pois_office)} offices, {len(pois_leisure)} leisure spots.")

Found: 223 schools, 4620 homes, 9 offices, 140 leisure spots.


/tmp/ipykernel_46942/3361663060.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_copy['geometry'] = gdf_copy['geometry'].centroid
/tmp/ipykernel_46942/3361663060.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_copy['geometry'] = gdf_copy['geometry'].centroid
/tmp/ipykernel_46942/3361663060.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_copy['geometry'] = gdf_copy['geometry'].centroid
/tmp/ipykernel_46942/3361663060.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-p

In [9]:
def get_random_poi(gdf):
    if gdf is None or gdf.empty: return None
    return gdf.sample(1).iloc[0].geometry

def generate_route_points(start_node, end_node, graph, start_time, speed_kmh=30):
    """Generates a list of (timestamp, lat, lon) tuples along a route."""
    route_points = []
    try:
        route_nodes = nx.shortest_path(graph, start_node, end_node, weight='length')
        path_geoms = nodes_proj.loc[route_nodes].geometry
        current_time = start_time
        
        for i in range(len(path_geoms) - 1):
            point_a, point_b = path_geoms.iloc[i], path_geoms.iloc[i+1]
            line_segment = LineString([point_a, point_b])
            distance_m = line_segment.length
            travel_time_s = (distance_m / 1000) / speed_kmh * 3600
            time_step = timedelta(seconds=20)
            num_steps = int(travel_time_s / time_step.total_seconds())
            
            if num_steps > 0:
                for j in range(num_steps + 1):
                    interp_point = line_segment.interpolate(j / num_steps, normalized=True)
                    offset_x, offset_y = random.uniform(-15, 15), random.uniform(-15, 15)
                    noisy_point = Point(interp_point.x + offset_x, interp_point.y + offset_y)
                    # FIX: Pass geometry as a positional argument
                    projected_geom, _ = ox.projection.project_geometry(noisy_point, crs=road_network_proj.graph['crs'], to_latlong=True)
                    lon, lat = projected_geom.x, projected_geom.y
                    route_points.append({"TimestampUTC": current_time + (j * time_step), "Latitude": lat, "Longitude": lon})
            
            current_time += timedelta(seconds=max(travel_time_s, 20))
        return route_points, current_time
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return [], start_time

def generate_wandering_points(center_point, start_time, duration_minutes=30, num_points=15):
    """Generates random points within a radius of a center point over a duration."""
    wandering_points = []
    if not center_point: return wandering_points, start_time
    time_increment = timedelta(minutes=duration_minutes / max(1, num_points))
    # Project the center point once at the beginning
    projected_center, _ = ox.projection.project_geometry(center_point, crs='epsg:4326', to_crs=road_network_proj.graph['crs'])
    
    for i in range(num_points):
        angle, radius = random.uniform(0, 2 * np.pi), random.uniform(0, 200)
        offset_x, offset_y = radius * np.cos(angle), radius * np.sin(angle)
        wandering_point_proj = Point(projected_center.x + offset_x, projected_center.y + offset_y)
        # FIX: Pass geometry as a positional argument
        projected_geom, _ = ox.projection.project_geometry(wandering_point_proj, crs=road_network_proj.graph['crs'], to_latlong=True)
        lon, lat = projected_geom.x, projected_geom.y
        wandering_points.append({"TimestampUTC": start_time + (i * time_increment), "Latitude": lat, "Longitude": lon})
        
    return wandering_points, start_time + timedelta(minutes=duration_minutes)

# --- Main Simulation Loop ---
all_movement_records = []
for user_id in range(NUM_SIMULATED_USERS):
    device_id = f"simulated_user_{user_id+1}"
    print(f"\nSimulating data for {device_id}...")
    
    persona_type = "student" if random.random() < 0.6 else "office_worker"
    
    home_location = get_random_poi(pois_home)
    work_school_location = get_random_poi(pois_school) if persona_type == "student" else get_random_poi(pois_office)
        
    if not all([home_location, work_school_location]):
        print(f"Skipping user {user_id+1} due to missing critical POIs.")
        continue
        
    home_node = ox.nearest_nodes(road_network, home_location.x, home_location.y)
    work_school_node = ox.nearest_nodes(road_network, work_school_location.x, work_school_location.y)
    
    start_date = datetime(datetime.now().year, 1, 1)

    for day in range(NUM_SIMULATION_DAYS):
        current_day = start_date + timedelta(days=day)
        
        if current_day.weekday() < 5 and random.random() < 0.9: 
            time_leave_home = current_day.replace(hour=random.randint(7, 8), minute=random.randint(0, 59))
            path, time_arrive_work = generate_route_points(home_node, work_school_node, road_network_proj, time_leave_home, speed_kmh=random.uniform(20, 45))
            for point in path:
                point["DeviceID"] = device_id
                all_movement_records.append(point)

            wandering_pts, current_time = generate_wandering_points(work_school_location, time_arrive_work, duration_minutes=random.randint(240, 480))
            for point in wandering_pts:
                point["DeviceID"] = device_id
                all_movement_records.append(point)

            if random.random() < 0.5:
                activity_choice = random.choice([pois_leisure, pois_commercial, pois_food])
                dest_poi = get_random_poi(activity_choice)
                if dest_poi:
                    dest_node = ox.nearest_nodes(road_network, dest_poi.x, dest_poi.y)
                    path, time_at_activity = generate_route_points(work_school_node, dest_node, road_network_proj, current_time, speed_kmh=random.uniform(20, 40))
                    for p in path: p["DeviceID"] = device_id; all_movement_records.append(p)

                    wandering_pts, current_time = generate_wandering_points(dest_poi, time_at_activity, duration_minutes=random.randint(60, 150))
                    for p in wandering_pts: p["DeviceID"] = device_id; all_movement_records.append(p)
                    
                    path_home, _ = generate_route_points(dest_node, home_node, road_network_proj, current_time, speed_kmh=random.uniform(20, 40))
                    for p in path_home: p["DeviceID"] = device_id; all_movement_records.append(p)
            else:
                path_home, _ = generate_route_points(work_school_node, home_node, road_network_proj, current_time, speed_kmh=random.uniform(25, 50))
                for p in path_home: p["DeviceID"] = device_id; all_movement_records.append(p)
        else:
            if random.random() < 0.7:
                time_leave_home = current_day.replace(hour=random.randint(10, 14), minute=random.randint(0, 59))
                activity_choice = random.choice([pois_leisure, pois_commercial, pois_food, pois_food])
                dest_poi = get_random_poi(activity_choice)
                if dest_poi:
                    dest_node = ox.nearest_nodes(road_network, dest_poi.x, dest_poi.y)
                    path, time_at_activity = generate_route_points(home_node, dest_node, road_network_proj, time_leave_home, speed_kmh=random.uniform(25, 50))
                    for p in path: p["DeviceID"] = device_id; all_movement_records.append(p)

                    wandering_pts, current_time = generate_wandering_points(dest_poi, time_at_activity, duration_minutes=random.randint(90, 240))
                    for p in wandering_pts: p["DeviceID"] = device_id; all_movement_records.append(p)

                    path_home, _ = generate_route_points(dest_node, home_node, road_network_proj, current_time, speed_kmh=random.uniform(25, 50))
                    for p in path_home: p["DeviceID"] = device_id; all_movement_records.append(p)



Simulating data for simulated_user_1...

Simulating data for simulated_user_2...

Simulating data for simulated_user_3...

Simulating data for simulated_user_4...

Simulating data for simulated_user_5...

Simulating data for simulated_user_6...

Simulating data for simulated_user_7...

Simulating data for simulated_user_8...

Simulating data for simulated_user_9...

Simulating data for simulated_user_10...

Simulating data for simulated_user_11...

Simulating data for simulated_user_12...

Simulating data for simulated_user_13...

Simulating data for simulated_user_14...

Simulating data for simulated_user_15...

Simulating data for simulated_user_16...

Simulating data for simulated_user_17...

Simulating data for simulated_user_18...

Simulating data for simulated_user_19...

Simulating data for simulated_user_20...

Simulating data for simulated_user_21...

Simulating data for simulated_user_22...

Simulating data for simulated_user_23...

Simulating data for simulated_user_24...



In [10]:
if all_movement_records:
    simulated_df = pd.DataFrame(all_movement_records)
    simulated_df['LocationID'] = -1
    simulated_df['Confidence'] = 100.0
    simulated_df['Description'] = "simulated_path"
    simulated_df['StatusCode'] = 0
    simulated_df['DBDatePublishedUTC'] = None
    simulated_df['EncryptedPayloadDB'] = None
    
    final_columns = [
        "LocationID", "DeviceID", "TimestampUTC", "Latitude", "Longitude",
        "Confidence", "Description", "StatusCode", "DBDatePublishedUTC",
        "EncryptedPayloadDB"
    ]
    simulated_df = simulated_df[final_columns]
    simulated_df.sort_values(by=['DeviceID', 'TimestampUTC'], inplace=True)
    
    output_path = os.path.join("./HistoricalMovement", OUTPUT_FILENAME)
    simulated_df.to_parquet(output_path, index=False)
    
    print(f"\nSuccessfully generated and saved {len(simulated_df)} data points to '{output_path}'")
    print("\nSample of generated data:")
    print(simulated_df.head())
else:
    print("\nNo data was generated. Please check POI availability or network connection.")


Successfully generated and saved 224218 data points to './HistoricalMovement/simulated_movement_data.parquet'

Sample of generated data:
   LocationID          DeviceID        TimestampUTC   Latitude   Longitude  \
0          -1  simulated_user_1 2025-01-01 10:25:00  16.052529  108.236235   
1          -1  simulated_user_1 2025-01-01 10:25:20  16.051803  108.233634   
2          -1  simulated_user_1 2025-01-01 10:25:40  16.051013  108.230880   
3          -1  simulated_user_1 2025-01-01 10:26:00  16.050538  108.228318   
4          -1  simulated_user_1 2025-01-01 10:26:20  16.049818  108.225795   

   Confidence     Description  StatusCode DBDatePublishedUTC  \
0       100.0  simulated_path           0               None   
1       100.0  simulated_path           0               None   
2       100.0  simulated_path           0               None   
3       100.0  simulated_path           0               None   
4       100.0  simulated_path           0               None   

  Encryp